# PET Bottle LCA

This is my working notebook for a cradle-to-grave carbon footprint of a 0.5 L PET bottle. Brightway does the calculation, and BAFU (the Swiss background database) supplies the supply chain data behind each stage.

I'm doing this in a notebook rather than a plain script for a few reasons. Building an LCA model is mostly trial and error: searching for the right background activity, checking whether an import actually linked correctly, tweaking one amount at a time. A notebook lets me run a step, look at what came back, then decide what to do next, instead of rerunning the whole thing every time something needs adjusting. That mattered a lot here, since importing BAFU threw a string of errors (missing fields, category naming that didn't match biosphere3) that had to be worked through one at a time.

Brightway itself has no GUI of its own to build a model in, it's a Python library, so a notebook is the natural place to write against it. Once the databases below are written, though, they show up in Activity Browser too, and from there on I use both: the notebook for building and debugging, Activity Browser for looking at results, Sankey diagrams, and Monte Carlo.

What's in here:
1. Project setup: create the Brightway project, install the elementary flows
2. Import BAFU: bring in the Swiss background database from ecoSpold1
3. Build the foreground: define the PET bottle itself, linked to BAFU
4. Run the LCA: calculate the climate change score for one bottle
5. Monte Carlo</cell id="70159cd9">

## 1. Project setup

Every Brightway calculation lives inside a project, its own isolated workspace with its own databases and settings (this is the same thing Activity Browser shows in its project sidebar). `bw2setup()` installs `biosphere3`, the roughly 4,000 standardised elementary flows (CO2, methane, land use, and so on) that every impact method and every background database ultimately refers back to. Nothing can be characterised into a score without it. It only needs to run once per project, running it again later is harmless, just unnecessary.

In [1]:
import bw2data
import bw2calc
import bw2io
from types import SimpleNamespace

bw = SimpleNamespace(
    projects=bw2data.projects,
    databases=bw2data.databases,
    methods=bw2data.methods,
    Database=bw2data.Database,
    get_activity=bw2data.get_activity,
    LCA=bw2calc.LCA,
    bw2setup=bw2io.bw2setup,
)

bw.projects.set_current("PET Bottle LCA")
bw.bw2setup()  # installs biosphere3, skip if already run for this project

print(list(bw.databases))

Biosphere database already present!!! No setup is needed
['biosphere3', 'BAFU', 'PET Bottle']


## 2. Import BAFU (ecoSpold1)

The bottle model needs background data, things like PET resin production, transport, and waste treatment, that I'm not measuring myself. BAFU (Bundesamt für Umwelt) provides the Swiss version of this data. It comes in the ecoSpold1 XML format (an older ecoinvent format), so it needs Brightway's dedicated `SingleOutputEcospold1Importer` rather than the newer ecospold2 or Excel importers.

Update the path below to wherever the extracted ecoSpold1 folder actually sits.

In [2]:
import bw2io as bi

ECOSPOLD_PATH = "/Users/abhi/ecoSpold files"

importer = bi.SingleOutputEcospold1Importer(ECOSPOLD_PATH, "BAFU")

Extracting XML data from 11947 datasets


/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(

Extracted 11947 datasets in 8.53 seconds


### 2a. Fix `None` fields before hashing

Some BAFU datasets have an explicit `None` sitting in fields the activity hash relies on, which breaks `set_code_by_activity_hash` further down. Clean those up first.

In [3]:
fields_to_clean = ["name", "categories", "unit", "reference product", "location"]

for ds in importer.data:
    for field in fields_to_clean:
        if ds.get(field) is None:
            ds[field] = ""
    for exc in ds.get("exchanges", []):
        for field in fields_to_clean:
            if exc.get(field) is None:
                exc[field] = ""

### 2b. Apply standard strategies

Raw ecoSpold1 XML isn't usable as-is. Units need normalising, activity codes need generating, and every exchange needs to be linked to something real in Brightway, either another BAFU activity or a `biosphere3` flow. `apply_strategies()` runs through bw2io's built-in cleanup pipeline (11 steps in total: normalising units, assigning production exchanges, hashing activity codes, and so on). `match_database("biosphere3", ...)` then tries to link every biosphere exchange (emissions, resource use) to its `biosphere3` counterpart by name and category. `statistics()` reports how many are still unresolved after that, which is what gets fixed next.

In [4]:
importer.apply_strategies()
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: normalize_units
Applying strategy: assign_only_product_as_production
Applying strategy: clean_integer_codes
Applying strategy: drop_unspecified_subcategories
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: update_ecoinvent_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: link_technosphere_by_activity_hash
Applied 11 strategies in 8.34 seconds
Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
295213 unlinked exchanges
  Type biosphere: 2679 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 295213)

### 2c. Fix BAFU's older category naming

Turns out BAFU's ecoSpold1 export uses an older, abbreviated way of naming categories:

| BAFU category | biosphere3 equivalent |
|---|---|
| `('emissions to air',)` | `('air',)` |
| `('emissions to air', 'high. pop.')` | `('air', 'urban air close to ground')` |
| `('emissions to air', 'low. pop.')` | `('air', 'non-urban air or from high stacks')` |
| `('emissions to soil',)` | `('soil',)` |
| `('emissions to water',)` | `('water',)` |
| `('resources', ...)` | `('natural resource', ...)` |

bw2io's built-in migration expects the `('air', 'low population density')` style of naming, so it never caught these. I map BAFU's raw spelling straight to biosphere3's final category names below, then re-run the matching.

In [5]:
CATEGORY_FIX = {
    ("emissions to air",): ("air",),
    ("emissions to air", "high. pop."): ("air", "urban air close to ground"),
    ("emissions to air", "low. pop."): ("air", "non-urban air or from high stacks"),
    ("emissions to air", "stratosphere + troposphere"): ("air", "lower stratosphere + upper troposphere"),
    ("emissions to soil",): ("soil",),
    ("emissions to soil", "agricultural"): ("soil", "agricultural"),
    ("emissions to soil", "industrial"): ("soil", "industrial"),
    ("emissions to water",): ("water",),
    ("emissions to water", "ground-"): ("water", "ground-"),
    ("emissions to water", "ocean"): ("water", "ocean"),
    ("emissions to water", "river"): ("water", "surface water"),
    ("resources",): ("natural resource",),
    ("resources", "land"): ("natural resource", "land"),
    ("resources", "in ground"): ("natural resource", "in ground"),
    ("resources", "in air"): ("natural resource", "in air"),
    ("resources", "in water"): ("natural resource", "in water"),
    ("resources", "biotic"): ("natural resource", "biotic"),
}

fixed_count = 0
for ds in importer.data:
    for exc in ds.get("exchanges", []):
        if exc.get("type") == "biosphere":
            cats = tuple(exc.get("categories") or ())
            if cats in CATEGORY_FIX:
                exc["categories"] = CATEGORY_FIX[cats]
                fixed_count += 1

print(f"Rewrote categories on {fixed_count} exchanges")

Rewrote categories on 268970 exchanges


### 2d. Re-run matching against biosphere3

In [6]:
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
114060 unlinked exchanges
  Type biosphere: 1155 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 114060)

### 2e. Inspect what's still unlinked

If the biosphere count above dropped sharply, and it should, most of the 2,679 unique flows were just the category naming mismatch, check what's left over.

In [7]:
unlinked = list(importer.unlinked)
bio_unlinked = [u for u in unlinked if u.get("type") == "biosphere"]
tech_unlinked = [u for u in unlinked if u.get("type") == "technosphere"]

print(f"{len(bio_unlinked)} unique unlinked biosphere flows")
for u in bio_unlinked[:20]:
    print(u.get("name"), "|", u.get("categories"), "|", u.get("unit"))

print(f"\n{len(tech_unlinked)} unique unlinked technosphere")
for u in tech_unlinked[:24]:
    print(u.get("name"), "|", u.get("location"), "|", u.get("unit"))

1155 unique unlinked biosphere flows
NMVOC, non-methane volatile organic compounds, unspecified origin | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'non-urban air or from high stacks') | kilogram
Occupation, industrial area, built up | ('natural resource', 'land') | square meter-year
Transformation, to industrial area, built up | ('natural resource', 'land') | square meter
Particulates, < 2.5 um | ('air', 'urban air close to ground') | kilogram
Copper | ('air', 'urban air close to ground') | kilogram
Particulates, < 2.5 um | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'urban air close to ground') | kilogram
Selenium | ('air', 'non-urban air or from high stacks') | kilogram
Selenium | ('air', 'urban air close to ground') | kilogram
Nickel | ('air', 'non-urban air or from high stacks') | kilogram
Zinc | ('air', 'non-urban air or from high stacks') | kilogram
Nickel | ('air', 'urban air close to ground') | kilogram
Zinc | ('air'

### 2f. Drop remaining unlinked exchanges and write the database

What's left unlinked (natural gas pipeline imports, direct air capture, concrete disposal) are BAFU processes that point at ecoinvent activities I don't have in this project. They won't resolve until ecoinvent is imported too. For now I drop the unlinked exchanges so the database can actually be written, they can be relinked later through Activity Browser's Relink database option once ecoinvent is added.

Only run the cell below once you've looked over the unlinked list above and are fine dropping them.

In [8]:
importer.drop_unlinked(i_am_reckless=True)
importer.statistics()

Applying strategy: drop_unlinked
Applied 1 strategies in 0.28 seconds
11947 datasets
306003 exchanges
0 unlinked exchanges
  


(11947, 306003, 0)

In [10]:
importer.write_database()
print(list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:50:03
  Finished: 08/06/2026 11:50:11
  Total time elapsed: 00:00:08
  CPU %: 94.90
  Memory %: 6.12
Created database: BAFU
['biosphere3', 'BAFU', 'PET Bottle']


## 3. Build the foreground: PET bottle

BAFU only gives me background processes, raw materials, generic transport, waste treatment. The bottle itself, the specific mix of resin, forming, distance, and end-of-life split that makes this particular product system, doesn't exist yet. That's the foreground model: a small database I write myself, where every exchange points at a real BAFU (or biosphere3) activity found by searching.

This is also the part that stays fully under my control. Mass values, distances, end-of-life shares, all of it can be changed and rerun any time without touching BAFU itself.

In [11]:
ei = bw.Database("BAFU")

# Search for the activities we need. Adjust search terms if BAFU's naming differs.
for r in ei.search("polyethylene terephthalate"):
    print(r, "|", r["location"])

'Fleece production, polyethylene terephthalate' (kilogram, RER, ['plastics']) | RER
'Polyethylene terephthalate, granulate, amorphous, at plant' (kilogram, RER, ['plastics', 'thermoplasts']) | RER
'Polyethylene terephthalate, granulate, bottle grade, at plant' (kilogram, RER, ['plastics', 'thermoplasts']) | RER
'xx Recycling PET' (kilogram, RER, ['waste treatment, obsolete', 'recycling, obsolete']) | RER
'Ethylene glycol, at plant' (kilogram, RER, ['chemicals', 'organic']) | RER


I search first rather than guess the exact name, because BAFU's naming doesn't always match what you'd expect (recycling shows up as `'xx Recycling PET'`, not `'Recycling of PET'`, for instance). `ei.search(...)` does a fuzzy text match across all activity names, so I can see the real options along with their `location` and `unit` before committing to one. Skipping this and guessing a name would just throw a `KeyError`.

In [12]:
for r in ei.search("blow moulding"):
    print(r, "|", r["location"])

'Stretch blow moulding' (kilogram, RER, ['plastics']) | RER
'xx Blow moulding' (kilogram, RER, ['processing, obsolete', 'plastics']) | RER
'Laptop computer, at plant' (unit, GLO, ['electronics', 'devices']) | GLO
'HDD, desktop computer, at plant' (unit, GLO, ['electronics', 'component']) | GLO
'CD-ROM, DVD-ROM drive, desktop computer, at plant' (unit, GLO, ['electronics', 'devices\\module\\component']) | GLO
'HDD, laptop computer, at plant' (unit, GLO, ['electronics', 'component']) | GLO
'CD-ROM, DVD-ROM drive, laptop computer, at plant' (unit, GLO, ['electronics', 'devices\\module\\component']) | GLO
'Desktop computer, without screen, at plant' (unit, GLO, ['electronics', 'devices']) | GLO


In [13]:
for r in ei.search("transport lorry"):
    print(r, "|", r["location"])

'Transport, municipal waste collection, lorry 21t' (ton kilometer, CH, ['transport systems', 'road']) | CH
'Transport, freight, lorry, fleet average' (ton kilometer, RER, ['transport systems', 'road\\Transformation']) | RER
'Transport, freight, lorry, fleet average' (ton kilometer, CH, ['transport systems', 'road\\Transformation']) | CH
'Lorry 16t' (unit, RER, ['transport systems', 'road\\operations\\infrastructure']) | RER
'Lorry 28t' (unit, RER, ['transport systems', 'road\\operations\\infrastructure']) | RER
'Lorry 40t' (unit, RER, ['transport systems', 'road\\operations\\infrastructure']) | RER
'Transport, freight, lorry, 3.5t gross weight, fleet average' (ton kilometer, CH, ['transport systems', 'road\\Transformation']) | CH
'Transport, freight, lorry, diesel, fleet average, long haul' (ton kilometer, CH, ['transport systems', 'road\\Transformation']) | CH
'Transport, freight, lorry, diesel, fleet average, urban delivery' (ton kilometer, RER, ['transport systems', 'road\\Transform

In [14]:
for r in ei.search("municipal solid waste incineration"):
    print(r, "|", r["location"])

'Disposal, municipal solid waste, 22.9% water, to municipal incineration' (kilogram, CH, ['waste management', 'municipal incineration']) | CH
'Disposal, LCD module,  to municipal waste incineration' (kilogram, CH, ['waste management', 'municipal incineration']) | CH
'Heat from waste, at municipal waste incineration plant' (megajoule, CH, ['heat', 'waste']) | CH
'Electricity from waste, at municipal waste incineration plant' (kilowatt hour, CH, ['electricity by fuel', 'waste']) | CH
'Disposal, coatings in CRT screens,  to municipal waste incineration' (kilogram, CH, ['waste management', 'municipal incineration']) | CH
'Carbon dioxide, captured at municipal solid waste incineration plant, for subsequent reuse' (kilogram, CH, ['chemicals', 'gases\\transformation']) | CH
'Disposal, wood and wood materials as construction waste, to municipal waste incineration, solid wood' (kilogram, CH, ['waste management', 'building demolition']) | CH
'Disposal, digester sludge, to municipal incineration'

In [15]:
for r in ei.search("sanitary landfill"):
    print(r, "|", r["location"])

'Disposal, paint, 0% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Disposal, aluminium, 0% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Process-specific burdens, sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Disposal, inert material, 0% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Disposal, tin sheet, 0% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Disposal, bitumen, 1.4% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Disposal, newspaper, 14.7% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Disposal, gypsum, 19.4% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Disposal, emulsion paint, 0% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Disposal, ash oli

In [16]:
for r in ei.search("recycling PET"):
    print(r, "|", r["location"])

'xx Recycling PET' (kilogram, RER, ['waste treatment, obsolete', 'recycling, obsolete']) | RER
'Disposal, flexible duct, aluminum/PET, DN of 125' (meter, CH, ['construction waste']) | CH
'Sink, chromium steel, at plant' (unit, CH, ['construction', 'kitchen']) | CH
'Disposal, polyethylene terephtalate, 0.2% water, to municipal incineration' (kilogram, CH, ['waste management', 'municipal incineration']) | CH


A note on why these particular picks, and why the exchange amounts below are formulas rather than fixed numbers:

`pet_resin` is bottle-grade PET specifically, not the generic amorphous kind, since the two have different production footprints. `blow_mould` is "stretch blow moulding", the actual process used to shape a PET bottle (injection moulding is for wide-mouth containers instead). `truck` is a fleet average rather than one specific vehicle, since I don't know the real distribution mix and a fleet average is the standard default. For `incineration` and `landfill`, BAFU has no PET-specific disposal dataset, so polyethylene stands in as a proxy, worth stating explicitly in a report since PE and PET don't incinerate identically, even though both are thermoplastics.

The mass balance, how much PET goes to recycling versus incineration versus landfill, is defined once at the top as `EoL_RECYCLE`, `EoL_INCIN`, `EoL_LANDFILL`. Every exchange amount below multiplies `PET_MASS_KG` by the relevant share, so changing the end-of-life scenario later just means editing three numbers instead of six.

In [17]:
pet_resin = ei.search("polyethylene terephthalate")[2]
print(pet_resin)  # 'Polyethylene terephthalate, granulate, bottle grade, at plant' (RER)

blow_mould = ei.search("blow moulding")[0]
print(blow_mould)  # 'Stretch blow moulding' (kilogram, RER)

truck = ei.search("transport lorry")[2]
print(truck)  # 'Transport, freight, lorry, fleet average' (ton kilometer, CH)

recycling = ei.search("recycling PET")[0]
print(recycling)  # 'xx Recycling PET' (kilogram, RER)

# Polyethylene used as proxy for PET disposal. BAFU has no PET-specific
# incineration/landfill dataset; PE is the accepted stand-in for thermoplastics.
incineration = ei.search("municipal solid waste incineration")[17]
print(incineration)  # 'Disposal, polyethylene, 0.4% water, to municipal incineration' (CH)

landfill = ei.search("sanitary landfill")[12]
print(landfill)  # 'Disposal, polyethylene, 0.4% water, to sanitary landfill' (CH)

'Polyethylene terephthalate, granulate, bottle grade, at plant' (kilogram, RER, ['plastics', 'thermoplasts'])
'Stretch blow moulding' (kilogram, RER, ['plastics'])
'Transport, freight, lorry, fleet average' (ton kilometer, CH, ['transport systems', 'road\\Transformation'])
'xx Recycling PET' (kilogram, RER, ['waste treatment, obsolete', 'recycling, obsolete'])
'Disposal, polyvinylfluoride, 0.2% water, to municipal incineration' (kilogram, CH, ['waste management', 'municipal incineration'])
'Disposal, paper, 11.2% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill'])


In [18]:
PET_MASS_KG = 0.025
DIST_KM = 100
EoL_RECYCLE = 0.48
EoL_INCIN = 0.37
EoL_LANDFILL = 0.15

DB_NAME = "PET Bottle"

if DB_NAME in bw.databases:
    del bw.databases[DB_NAME]

foreground = bw.Database(DB_NAME)
foreground.write({
    (DB_NAME, "pet_bottle_0.5L"): {
        "name": "PET bottle, 0.5 litre, cradle-to-grave",
        "reference product": "PET bottle, 0.5 litre, cradle-to-grave",
        "unit": "piece",
        "location": "CH",
        "type": "process",
        "exchanges": [
            {"input": (DB_NAME, "pet_bottle_0.5L"), "amount": 1, "unit": "piece", "type": "production"},
            {"input": pet_resin.key, "amount": PET_MASS_KG, "unit": "kilogram", "type": "technosphere"},
            {"input": blow_mould.key, "amount": PET_MASS_KG, "unit": "kilogram", "type": "technosphere"},
            {"input": truck.key, "amount": PET_MASS_KG * DIST_KM / 1000, "unit": "ton kilometer", "type": "technosphere"},
            {"input": recycling.key, "amount": PET_MASS_KG * EoL_RECYCLE, "unit": "kilogram", "type": "technosphere"},
            {"input": incineration.key, "amount": PET_MASS_KG * EoL_INCIN, "unit": "kilogram", "type": "technosphere"},
            {"input": landfill.key, "amount": PET_MASS_KG * EoL_LANDFILL, "unit": "kilogram", "type": "technosphere"},
        ]
    }
})

print("Database written:", list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:51:11
  Finished: 08/06/2026 11:51:11
  Total time elapsed: 00:00:00
  CPU %: 95.30
  Memory %: 0.68
Database written: ['biosphere3', 'BAFU', 'PET Bottle']


## 4. Run the LCA

`{bottle: 1}` is the functional unit, it tells Brightway to calculate the impact of making exactly one of this activity. Every LCA needs this stated explicitly, otherwise "impact of a bottle" is ambiguous (impact of making it? of a whole batch? of just the resin?).

The method tuple picks which set of characterisation factors to apply. Different methods answer different questions (climate change versus acidification versus water use), and the tuple has to match exactly what's installed, which is why I list `bw.methods` first rather than guessing.

`lca.lci()` builds and solves the inventory, tracing the whole supply chain (bottle, resin, crude oil, and so on) and tallying every elementary flow released along the way. `lca.lcia()` then characterises that inventory, multiplying each flow by its impact factor and summing to one score. `lca.score` is that number: kg CO2-equivalent per bottle, comparable across products and studies using the same method.

In [19]:
# List available methods to find the exact name before picking one:
[m for m in bw.methods if "IPCC" in str(m)]

[('IPCC 2013 no LT',
  'climate change no LT',
  'global temperature change potential (GTP100) no LT'),
 ('IPCC 2013 no LT',
  'climate change no LT',
  'global temperature change potential (GTP20) no LT'),
 ('IPCC 2013 no LT',
  'climate change no LT',
  'global warming potential (GWP100) no LT'),
 ('IPCC 2013 no LT',
  'climate change no LT',
  'global warming potential (GWP20) no LT'),
 ('IPCC 2013',
  'climate change',
  'global temperature change potential (GTP100)'),
 ('IPCC 2013',
  'climate change',
  'global temperature change potential (GTP20)'),
 ('IPCC 2013', 'climate change', 'global warming potential (GWP100)'),
 ('IPCC 2013', 'climate change', 'global warming potential (GWP20)'),
 ('IPCC 2021 no LT',
  'climate change no LT',
  'global temperature change potential (GTP100) no LT'),
 ('IPCC 2021 no LT',
  'climate change no LT',
  'global temperature change potential (GTP50) no LT'),
 ('IPCC 2021 no LT',
  'climate change no LT',
  'global warming potential (GWP100) no LT

In [20]:
bottle = bw.get_activity((DB_NAME, "pet_bottle_0.5L"))

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

lca = bw.LCA({bottle: 1}, method)
lca.lci()
lca.lcia()

print(f"GWP 100a: {lca.score:.4f} kg CO2-eq per bottle")

GWP 100a: 0.0880 kg CO2-eq per bottle


## 5. Monte Carlo uncertainty analysis

The single `GWP 100a` score above is one point estimate, and it hides how much that result could actually vary given real uncertainty in every exchange behind it. Monte Carlo reruns the same calculation many times, resampling every exchange from its uncertainty distribution each time, and returns a spread of scores instead of one number. A mean plus a confidence interval is a more honest thing to report than a single figure, especially once you're comparing products.

Two things worth knowing before running this:

`bw2calc`'s Monte Carlo solver can raise `ValueError: 'scipy.sparse.linalg.cgs' called with invalid atol='legacy'` on newer `scipy` versions. If that happens, run `conda install "scipy<1.12"` in the `ab` environment, restart the kernel, and re-run this cell.

Monte Carlo only produces a real spread if the exchanges being sampled actually carry uncertainty data. Background BAFU exchanges might carry some from the original ecoSpold1 data. The foreground exchanges I wrote myself (the bottle's own amounts) don't have uncertainty set by default, so they stay fixed across every iteration unless I add it explicitly, for example through Activity Browser's Uncertainty tab.

In [21]:
from bw2calc import MonteCarloLCA

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

mc = MonteCarloLCA({bottle: 1}, method)
n_iterations = 1000
scores = [next(mc) for _ in range(n_iterations)]

import numpy as np

print(f"Iterations: {n_iterations}")
print(f"Mean:   {np.mean(scores):.4f} kg CO2-eq per bottle")
print(f"Median: {np.median(scores):.4f} kg CO2-eq per bottle")
print(f"Std dev: {np.std(scores):.4f}")
print(f"5th to 95th percentile: {np.percentile(scores, 5):.4f} to {np.percentile(scores, 95):.4f} kg CO2-eq")

Iterations: 1000
Mean:   0.0955 kg CO2-eq per bottle
Median: 0.0911 kg CO2-eq per bottle
Std dev: 0.0269
5th to 95th percentile: 0.0623 to 0.1479 kg CO2-eq
